In [1]:
import pandas as pd
import numpy as np
import geopandas as gpd
import json
import sparrow as sp
import spatialdata as sd
from shapely.geometry import box, Point

the value of the environment variable BASIC_DCT_BACKEND is not in ["JAX","SCIPY"]
2025-09-22 13:25:20,667 - sparrow.image.pixel_clustering._clustering - WARNING - 'flowsom' not installed, 'sp.im.flowsom' will not be available.
2025-09-22 13:25:20,700 - sparrow.table.cell_clustering._clustering - WARNING - 'flowsom' not installed, 'sp.tb.flowsom' will not be available.
2025-09-22 13:25:20,722 - sparrow.plot - WARNING - 'joypy' not installed, 'sp.pl.ridgeplot_channel' and 'sp.pl.ridgeplot_channel_sample' will not be available.
2025-09-22 13:25:20,723 - sparrow.plot - WARNING - 'textalloc' not installed, 'sp.pl.group_snr_ratio' and 'sp.pl.snr_ratio' will not be available.


In [32]:
def get_containing_barcode(bins=(8, 16),
                           data_path = "../data/vizgen_polyT_45056_solved/"):
    
    sdata = sd.read_zarr(data_path)

    joined_df = gpd.sjoin(gpd.GeoDataFrame(geometry = sdata.shapes[f"grid_{bins[0]}um"].centroid),sdata.shapes[f"grid_{bins[1]}um"],
                    how='inner', predicate='within', lsuffix=str(bins[0]), rsuffix=str(bins[1]))
    joined_df[f'index_{bins[0]}'] = joined_df.index

    grid_8um = sdata.shapes[f"grid_{bins[0]}um"]
    grid_8um['containing_barcode'] = joined_df[f'index_{bins[1]}']

    # Filter out points that are not contained in any geometry
    contained_points = grid_8um[grid_8um['containing_barcode'].notnull()]
    print(f"All points in {bins[0]}um are contained in {bins[1]}um:", contained_points.shape[0] == grid_8um.shape[0])

    # Save tissue_positions_8 with containing_barcode to a new parquet file
    joined_df.rename(columns={f'index_{bins[1]}': 'containing_barcode', f'index_{bins[0]}': 'barcode'}, inplace=True)
    joined_df.drop(columns=['geometry']).to_parquet(f"/home/chananchidas/visium-hd/visium_hd_liver_combined/vizgen/tissue_positions_{bins[0]}um_{bins[1]}um.parquet")

In [ ]:
get_containing_barcode(bins=(8,16))

/home/chananchidas/miniconda3/envs/harpy/lib/python3.10/site-packages/zarr/creation.py:614: UserWarning: ignoring keyword argument 'read_only'
  compressor, fill_value = _kwargs_compat(compressor, fill_value, kwargs)
/tmp/ipykernel_54200/99391034.py:4: DeprecationWarning: Table group found in zarr store at location /home/chananchidas/visium-hd/data/vizgen_polyT_45056_solved. Please update the zarr store to use tables instead.
  sdata = sd.read_zarr(data_path)


All points in 8um are contained in 16um: True


#### Step-by-step

In [2]:
sdata = sd.read_zarr("../data/vizgen_polyT_45056_solved/")
sdata

/home/chananchidas/miniconda3/envs/harpy/lib/python3.10/site-packages/zarr/creation.py:614: UserWarning: ignoring keyword argument 'read_only'
  compressor, fill_value = _kwargs_compat(compressor, fill_value, kwargs)
/tmp/ipykernel_54200/119979360.py:1: DeprecationWarning: Table group found in zarr store at location /home/chananchidas/visium-hd/data/vizgen_polyT_45056_solved. Please update the zarr store to use tables instead.
  sdata = sd.read_zarr("../data/vizgen_polyT_45056_solved/")


SpatialData object, with associated Zarr store: /home/chananchidas/visium-hd/data/vizgen_polyT_45056_solved
├── Images
│     ├── 'clahe': DataArray[cyx] (1, 45056, 45056)
│     ├── 'raw_image': DataArray[cyx] (1, 45056, 45056)
│     └── 'tophat_filtered': DataArray[cyx] (1, 45056, 45056)
├── Labels
│     ├── 'grid_8um_labels': DataArray[yx] (45056, 45056)
│     ├── 'grid_16um_labels': DataArray[yx] (45056, 45056)
│     ├── 'grid_32um_labels': DataArray[yx] (45056, 45056)
│     └── 'segmentation_mask': DataArray[yx] (45056, 45056)
├── Points
│     └── 'transcripts': DataFrame with shape: (<Delayed>, 3) (2D points)
├── Shapes
│     ├── 'filtered_low_counts_segmentation_mask_boundaries': GeoDataFrame shape: (5, 1) (2D shapes)
│     ├── 'filtered_segmentation_mask_boundaries_low_counts': GeoDataFrame shape: (5409, 1) (2D shapes)
│     ├── 'filtered_segmentation_mask_boundaries_segmentation': GeoDataFrame shape: (3080, 1) (2D shapes)
│     ├── 'filtered_segmentation_mask_boundaries_size': G

In [6]:
sdata.shapes['grid_8um'].centroid

1           POINT (37.037 37.037)
2          POINT (111.111 37.037)
3          POINT (185.185 37.037)
4          POINT (259.259 37.037)
5          POINT (333.333 37.037)
                   ...           
369660    POINT (44703.704 45000)
369661    POINT (44777.778 45000)
369662    POINT (44851.852 45000)
369663    POINT (44925.926 45000)
369664        POINT (45000 45000)
Length: 369664, dtype: geometry

In [7]:
sdata.shapes['grid_16um']

,geometry
1,"POLYGON ((0 0, 148.148 0, 148.148 148.148, 0 1..."
2,"POLYGON ((148.148 0, 296.296 0, 296.296 148.14..."
3,"POLYGON ((296.296 0, 444.444 0, 444.444 148.14..."
4,"POLYGON ((444.444 0, 592.593 0, 592.593 148.14..."
5,"POLYGON ((592.593 0, 740.741 0, 740.741 148.14..."
...,...
92412,"POLYGON ((44296.296 44888.889, 44444.444 44888..."
92413,"POLYGON ((44444.444 44888.889, 44592.593 44888..."
92414,"POLYGON ((44592.593 44888.889, 44740.741 44888..."
92415,"POLYGON ((44740.741 44888.889, 44888.889 44888..."


In [ ]:
gpd.GeoDataFrame(geometry = sdata.shapes['grid_8um'].centroid)

,geometry
1,POINT (37.037 37.037)
2,POINT (111.111 37.037)
3,POINT (185.185 37.037)
4,POINT (259.259 37.037)
5,POINT (333.333 37.037)
...,...
369660,POINT (44703.704 45000)
369661,POINT (44777.778 45000)
369662,POINT (44851.852 45000)
369663,POINT (44925.926 45000)


In [17]:
joined_df = gpd.sjoin(gpd.GeoDataFrame(geometry = sdata.shapes['grid_8um'].centroid),sdata.shapes['grid_16um'],
                   how='inner', predicate='within', lsuffix='8', rsuffix='16')

grid_8um = sdata.shapes['grid_8um']
# Add the containing barcode to tissue_positions_8
grid_8um['containing_barcode'] = joined_df['index_16']

# Filter out points that are not contained in any geometry
contained_points = grid_8um[grid_8um['containing_barcode'].notnull()]
print("All points in 8um are contained in 16um:", contained_points.shape[0] == grid_8um.shape[0])


All points in 8um are contained in 16um: True


In [22]:
joined_df['index_8'] = joined_df.index

In [23]:
joined_df

,geometry,index_16,index_8
1,POINT (37.037 37.037),1,1
2,POINT (111.111 37.037),1,2
3,POINT (185.185 37.037),2,3
4,POINT (259.259 37.037),2,4
5,POINT (333.333 37.037),3,5
...,...,...,...
369660,POINT (44703.704 45000),92414,369660
369661,POINT (44777.778 45000),92415,369661
369662,POINT (44851.852 45000),92415,369662
369663,POINT (44925.926 45000),92416,369663
